# GLM-5.3-Flash UD-IQ2_XXS on 1× DGX Spark (GB10) — 262k context, llama.cpp

**Verdict: fits on one Spark with the full 262,144-token context, but slow — 15.1 tok/s decode, 245k-token prompt needs 28.8 min to prefill. One-node fallback, not the throughput pick.** Quality at 2 bits: untested.

In [1]:
# --- Status cell -------------------------------------------------------
# LIVE = False replays the committed receipts under results/<experiment>/.
# LIVE = True runs the same harness against a running endpoint whose URL
# comes from the environment. Never commit a notebook executed with LIVE = True.
import os, json
EXPERIMENT = "2026-10-01-glm-5.3-flash-ud-iq2xxs-1node-llamacpp"
RESULTS_DIR = os.path.join("..", "results", EXPERIMENT)
RECEIPTS = os.path.join(RESULTS_DIR, "receipts")
LIVE = False
print(f"experiment : {EXPERIMENT}\nLIVE       : {LIVE}")
if LIVE and not os.environ.get("GLM_ENDPOINT"):
    raise RuntimeError("LIVE=True but GLM_ENDPOINT is not set")
def receipt(name):
    with open(os.path.join(RECEIPTS, name)) as f:
        return json.load(f)

experiment : 2026-10-01-glm-5.3-flash-ud-iq2xxs-1node-llamacpp
LIVE       : False


## 1. TL;DR

| Metric | Value | Label |
|---|---|---|
| Served context | **262,144 tokens** (model native max 1,048,576) | measured |
| Decode, short prompt | 15.1 tok/s (thinking off), 15.7 (on) | measured |
| Decode at 28k / 128k / 245k context | 13.9 / 10.7 / 8.3 tok/s | measured (n=1 for 128k, 245k) |
| Prefill | 378 tok/s @ 7k, 346 @ 28k, 212 @ 128k, 141 @ 245k | measured |
| TTFT | 19.6 s @ 7k, 80 s @ 28k, 10 min @ 128k, 28.8 min @ 245k | measured |
| Memory | 106 GB used after load; 114–115 of 119 GB at 245k (~4 GB free) | measured |
| Quality at 2 bits | not scored | untested |
| Vision | not loaded | untested |

| Pin | Value |
|---|---|
| Model | `unsloth/GLM-5.3-Flash-GGUF` @ `621d456e…`, UD-IQ2_XXS (97 GB) |
| Runtime | `unslothai/llama.cpp` `glm5next/upstream` @ `86ebfef2…` (ggml-org PR #27754, open) |
| Stock llama.cpp b11277 | fails: `unknown model architecture: 'glm5next'` |
| Hardware | 1 × DGX Spark, driver 580.159.03, CUDA 13.0 |
| KV cache | q8_0 (indexer keys f16), 1 slot, no speculative decoding |

## 2. Visible results

In [2]:
b = receipt("bench.json")
rows = []
for k, v in {**b["short_and_mid"], **b["long"]}.items():
    rows.append((k, v.get("prompt_tokens"), v.get("ttft_s"), v.get("prefill_tok_s"), v.get("decode_tok_s")))
print(f"{'workload':28}{'prompt tok':>11}{'TTFT s':>9}{'prefill t/s':>13}{'decode t/s':>12}")
for k, p, t, pf, d in rows:
    f = lambda x, n=1: "-" if x is None else f"{x:.{n}f}"
    print(f"{k:28}{p:>11.0f}{f(t):>9}{f(pf, 0):>13}{f(d):>12}")
print("\nmethod:", b["method"])

workload                     prompt tok   TTFT s  prefill t/s  decode t/s
decode_short_think_off               33      0.7            -        15.1
decode_short_think_on                22      0.4            -        15.7
prefill_7k                         7398     19.6          378           -
prefill_28k                       27736     80.0          346           -
decode_at_28k_think_off           27850     78.9            -        13.9
ctx_128k                         127533    601.4          212        10.7
ctx_245k                         244570   1729.6          141         8.3

method: temperature 0, median of 3 (n=1 for 128k/245k), usage-object token counts, unique random prefix per request (no prompt cache hit), client on a separate machine over the network


## 3. Reproduce

Hardware: one DGX Spark with at least 119 GB usable memory and nothing else running on it. Files are in `results/2026-10-01-glm-5.3-flash-ud-iq2xxs-1node-llamacpp/harness/`.

1. Weights: `hf download unsloth/GLM-5.3-Flash-GGUF --include "UD-IQ2_XXS/*" --local-dir ~/glm53-gguf` (97 GB).
2. Image: `docker build -t glm53-llamacpp:unsloth-pr27754 -f Dockerfile .` (builds the open PR branch; ~5 min on the Spark; needs the CUDA stub-library link fix in the Dockerfile).
3. Serve: `serve.sh` (container with `--restart unless-stopped`, OpenAI-compatible API on :8888, model id `glm-5.3-flash`). Load takes ~95 s.
4. Measure: `python3 harness/bench.py http://HOST:8888/v1 glm-5.3-flash out.json`, then `long.py` for the 128k and 245k prompts.

The 245k prompt takes ~29 min to prefill; budget client timeouts accordingly.

In [3]:
env = receipt("env.json")
for k, v in env.items():
    print(f"{k}: {v}")

hardware: 1x NVIDIA DGX Spark (GB10, SM121, arm64, 128 GB unified memory), nothing else running on the node, driver 580.159.03, CUDA 13.0
model: {'checkpoint': 'unsloth/GLM-5.3-Flash-GGUF', 'revision': '621d456e93e926e4b52f85cff5f634358c1828f9', 'file': 'UD-IQ2_XXS (4 shards, 97 GB on disk)', 'base': 'zai-org/GLM-5.3-Flash (321B MoE, native max position 1,048,576)'}
runtime: {'engine': 'llama.cpp, unslothai/llama.cpp branch glm5next/upstream (ggml-org PR #27754, open)', 'commit': '86ebfef2c6a0f3359a2a07d2c215d61b0fa885c9', 'build': 'nvidia/cuda:13.0.1-devel-ubuntu24.04, CMAKE_CUDA_ARCHITECTURES=121, Release', 'stock_server_cuda13_b11277': "fails: unknown model architecture 'glm5next'"}
flags: -ngl 999 -c 262144 -np 1 -fa on -ctk q8_0 -ctv q8_0 -b 2048 -ub 1024 --jinja (indexer key cache stays f16)
served_context_tokens: 262144
container_restarts_after_all_tests: 0
speculative_decoding: none
memory: host 106 GB used after load, 114-115 GB of 119 GB after the 245k prompt (about 4 GB free

## Appendix: failures and limits

- Stock `ghcr.io/ggml-org/llama.cpp:server-cuda13` (build 11277) cannot load the GGUF (`unknown model architecture: 'glm5next'`); the open PR branch is required.
- First image build failed at link time (`undefined reference to cuGetErrorString`); fixed by linking against the CUDA stub `libcuda`.
- `--no-mmap` / `--mmap off` are not accepted by this build; default mmap was used.
- About 4 GB of host memory remains at full context. Do not co-locate another job.
- The 4-bit EXL3 build (vLLM, two Sparks, [cookbook](../docs/cookbook/glm-5.3-flash-dgx-spark.md)) is faster at decode; this run exists because a second node was not available.